# LINE Bot LoRA Fine-tune on Kaggle (Free P100/T4)

Kaggle 永久免費 30hr/wk GPU。Modal $30 免費額度用完後的替代方案。

**前置作業**：
1. 在本機跑 `python finetune/kaggle_upload.py` 把 `distilled.jsonl` 推上 Kaggle Datasets
2. 上 https://www.kaggle.com/code 開新 Notebook，**Settings → Accelerator** 選 GPU P100（或 T4 x2）
3. 右側 **Add Data** → 找到 `linebot-distilled` dataset 加進來
4. 上傳這個 .ipynb（File → Import Notebook）
5. 按 **Run All**（首次跑要 2-3 hr）
6. 跑完從右側 **Output** 下載 `adapter.zip`
7. 本機跑 `python finetune/kaggle_download_adapter.py` 整合 + 跑 acceptance gate

## Cell 1: 安裝依賴

In [ ]:
!pip install -q transformers peft accelerate datasets bitsandbytes
!pip list | grep -E '(transformers|peft|accelerate|datasets|bitsandbytes|torch)'

## Cell 2: 拉 Kaggle Dataset 的 distilled.jsonl

Kaggle 把 dataset 自動 mount 到 `/kaggle/input/<dataset-slug>/`。
預設 slug 是 `linebot-distilled`（kaggle_upload.py 寫死）。

In [ ]:
import json
import os
import random
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/linebot-distilled")
if not INPUT_DIR.exists():
    # fallback：找 /kaggle/input 下任一含 distilled.jsonl 的子目錄
    for child in Path("/kaggle/input").iterdir() if Path("/kaggle/input").exists() else []:
        if (child / "distilled.jsonl").exists():
            INPUT_DIR = child
            print(f"[INFO] auto-detected input dir: {INPUT_DIR}")
            break

DISTILLED = INPUT_DIR / "distilled.jsonl"
assert DISTILLED.exists(), f"distilled.jsonl not found at {DISTILLED} -- 確認右側 Add Data 加了 linebot-distilled"

pairs = []
with DISTILLED.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        # 丟 mock pair（沒有真實 assistant 回覆）
        if obj.get("metadata", {}).get("mock", False):
            continue
        pairs.append(obj)

print(f"[INFO] loaded {len(pairs)} non-mock pairs")

# 80/20 split (seed=42 跟 eval_harness 對齊)
random.seed(42)
indices = list(range(len(pairs)))
random.shuffle(indices)
cut = max(1, int(len(pairs) * 0.8))
train_idx = set(indices[:cut])
train_pairs = [p for i, p in enumerate(pairs) if i in train_idx]
eval_pairs = [p for i, p in enumerate(pairs) if i not in train_idx]
print(f"[INFO] train={len(train_pairs)}  eval={len(eval_pairs)}")

if len(train_pairs) < 100:
    print("[WARN] 訓練資料 < 100 筆，fine-tune 預期效果差。建議先累積到 1000+ 再來。")

## Cell 3: 載入 Qwen2.5-14B-Instruct

Kaggle P100 16GB / T4 16GB → 4-bit quant（QLoRA）剛好 fits 14B + LoRA。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 14B base — Kaggle P100 16GB / T4 16GB 4-bit OK；對齊本機 mlx + Modal cloud
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # for gradient checkpointing
print(f"[OK] loaded {MODEL_NAME}")
print(f"     trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Cell 4: LoRA config（rank=16, alpha=32, q/k/v/o, dropout=0.05）

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

# 14B 適配：rank 從 16 降到 8（base 大 5x）；alpha = 2 * rank
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 5: 訓練（epochs=3, batch=4, lr=2e-4）

In [ ]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

MAX_LEN = 2048

def format_pair(pair):
    """用 chat_template 把 messages 變成 single prompt+response 字串。"""
    msgs = pair["messages"]
    # 加 system 對齊 inference
    full_msgs = [{"role": "system", "content": "你是 LINE 群組對話助理咪寶，繁體中文簡短回覆，有具體觀點。"}] + msgs
    text = tokenizer.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False)
    return text

def to_features(pair):
    text = format_pair(pair)
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, padding=False)
    enc["labels"] = enc["input_ids"].copy()
    return enc

train_ds = Dataset.from_list(train_pairs).map(to_features, remove_columns=["messages", "metadata"])
eval_ds = Dataset.from_list(eval_pairs).map(to_features, remove_columns=["messages", "metadata"])

OUTPUT_DIR = "/kaggle/working/adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

trainer.train()
print("[OK] training done")

## Cell 6: 儲存 adapter weights → zip

In [ ]:
import shutil

ADAPTER_FINAL = "/kaggle/working/adapter_final"
trainer.model.save_pretrained(ADAPTER_FINAL)
tokenizer.save_pretrained(ADAPTER_FINAL)

ZIP_PATH = "/kaggle/working/adapter"
shutil.make_archive(ZIP_PATH, "zip", ADAPTER_FINAL)
print(f"[OK] zipped to {ZIP_PATH}.zip")

import os
size_mb = os.path.getsize(f"{ZIP_PATH}.zip") / (1024 * 1024)
print(f"     size: {size_mb:.1f} MB")
print("[NEXT] 跑完後到右側 Output panel 下載 adapter.zip，或 publish notebook 後用 kaggle_download_adapter.py 抓")

## Cell 7: Eval（從 distilled 切 20% 算 loss）

In [ ]:
metrics = trainer.evaluate()
print("[EVAL]", metrics)

# 儲存 metrics 給 acceptance gate 用
import json
EVAL_OUT = "/kaggle/working/eval_metrics.json"
with open(EVAL_OUT, "w", encoding="utf-8") as f:
    json.dump({
        "label": "kaggle_adapter",
        "eval_loss": metrics.get("eval_loss"),
        "eval_runtime": metrics.get("eval_runtime"),
        "eval_samples": len(eval_pairs),
        "train_samples": len(train_pairs),
        "model": MODEL_NAME,
        "lora": {"r": 8, "alpha": 16, "dropout": 0.05, "target_modules": ["q_proj","k_proj","v_proj","o_proj"]},
        "hyperparams": {"epochs": 3, "batch_size": 4, "lr": 2e-4},
    }, f, ensure_ascii=False, indent=2)
print(f"[OK] saved {EVAL_OUT}")
print("[NOTE] 完整的 4-metric eval（violation / chinese / rule0 / judge）要本機跑 eval_harness.py，因為 judge 需要 Gemini API key")